# 🎯 [ISR03] 드론 비행로그 패턴 이상 탐지와 운용 주체 단정 방지
### Google Colab 실습 환경 — 미션 07 (isr03_m07_student.ipynb) (정보·감시·정찰(ISR) / Level 3)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/ISR03/ISR03_solution.ipynb)

---

## 📌 1. 프로젝트 개요 및 배경

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 정보·감시·정찰(ISR) (Level 3) |
| **학습 소요 시간** | 10개 미션 (5일 / 총 27.5시간) |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
러시아-우크라이나 전쟁에서는 상용 드론이 정찰과 공격에 폭넓게 쓰이면서, 노획하거나 수집한 드론 비행로그를 분석해 이상 비행을 찾아내는 일이 중요해졌다. 그러나 같은 상용 기체를 아군·민간·적대 세력이 모두 쓰기 때문에, 비행 특성만 보고 운용 주체를 성급하게 단정하면 오판으로 이어질 수 있다. 본 프로젝트에서는 교육용 가상 비행로그를 코드로 직접 만들어 스키마 정의, GPS 스푸핑 보정 전처리, 규칙 기반 Baseline, 군집·비지도 학습 기반 이상 탐지, 불확실성을 반영한 운용 주체 신뢰도 엔진, 분석 대시보드, 정보분석관 심층 검토(HITL), 대량 로그 부하 시험, AAR 재보정, 최종 통합까지 단계별로 구현한다. 최종 목표는 이상 비행을 놓치지 않으면서 오탐을 줄이고, 운용 주체에 관한 판단은 항상 확률·불확실성·근거와 함께 사람의 검토를 거쳐 보고되도록 설계하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

---

### 🛠️ 2. 실습 실행 및 산출물 저장·데이터 연계 가이드

1. **Colab / 로컬 실행 환경 안내**
   - 아래 환경 설정 셀의 `!pip install -q ...` 구문은 **Google Colab / Jupyter 노트북 전용** 매직 커맨드입니다.
   - 일반 로컬 Python(`.py`) 환경에서 실행할 때는 해당 줄을 제거하고 터미널에서 `pip install numpy pandas matplotlib scipy scikit-learn`을 먼저 실행하세요.

2. **산출물(`deliverables`) 디렉토리 구조 및 파일 저장 방법**
   - 본 노트북의 각 미션 셀은 Colab 환경에서 단독 실행이 가능하도록 인메모리 시뮬레이션으로 구성되어 있습니다.
   - 프로젝트 제출용 산출물(`data/schema/`, `docs/`, `src/preprocess/`, `src/models/`, `src/verification/`, `src/ui/`, `src/hitl/`, `tests/`, `configs/`)을 실제 파일로 저장하려면, 환경 설정 셀에 제공된 `init_deliverable_dirs()` 및 `save_deliverable(rel_path, content)` 헬퍼 함수 또는 셀 상단의 `%%writefile <파일경로>` 매직 커맨드를 사용하세요.

3. **첨부 데이터셋(`isr03_mXX_dataset.json`) 연계 활용**
   - 각 미션 셀은 외부 파일 의존성 없이도 독립 실행되도록 자체 난수 기반 가상 데이터를 생성합니다.
   - 미션별 첨부파일(`isr03_m01_dataset.json` ~ `isr03_m10_dataset.json`, `isr03_mXX_sample.md`)은 `load_mission_dataset(1)` 함수로 불러와 스키마 대조, 경계값 추가 검증, 파라미터 확장 실험에 연계 활용할 수 있습니다.

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")

# 산출물 디렉토리 초기화 및 파일 저장/첨부 데이터셋 로드 헬퍼
DELIVERABLE_DIRS = [
    'data/schema', 'docs', 'src/preprocess', 'src/models',
    'src/verification', 'src/ui', 'src/hitl', 'tests', 'configs'
]

def init_deliverable_dirs(base_dir='.'):
    """프로젝트 필수 산출물 디렉토리를 생성한다."""
    for d in DELIVERABLE_DIRS:
        os.makedirs(os.path.join(base_dir, d), exist_ok=True)

def save_deliverable(rel_path, content, base_dir='.'):
    """산출물 파일(JSON, YAML, MD, PY 등)을 지정 경로에 저장한다."""
    full_path = os.path.join(base_dir, rel_path)
    os.makedirs(os.path.dirname(full_path), exist_ok=True)
    with open(full_path, 'w', encoding='utf-8') as f:
        f.write(content if isinstance(content, str) else json.dumps(content, ensure_ascii=False, indent=2))
    return full_path

def load_mission_dataset(mission_num, base_dir='.'):
    """첨부된 미션별 가상 데이터셋(isr03_mXX_dataset.json)을 불러온다."""
    fname = f'isr03_m{int(mission_num):02d}_dataset.json'
    for candidate in [os.path.join(base_dir, fname), os.path.join(base_dir, 'uploads', 'ISR03', fname)]:
        if os.path.exists(candidate):
            with open(candidate, encoding='utf-8') as f:
                return json.load(f)
    raise FileNotFoundError(f'{fname} 파일을 찾을 수 없습니다. Colab에 업로드했는지 확인하세요.')


---

## 🎯 미션 07. 운용 주체 최종 판정을 위한 정보분석관 심층 검토(HITL) 워크플로우 설계
**소요시간**: 약 3.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

드론 이상 징후 분석 결과에 대해 정보분석관이 출처와 로그 무결성을 확인하고 최종 분석 의견을 채택/보류하는 HITL 워크플로우를 설계한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **이상 징후 분석 결과를 정보분석관이 출처와 로그 무결성을 확인한 뒤 채택·보류·반려하는 인간 개입(HITL) 심층 검토 워크플로우를 설계한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **엔진 결과는 모두 검토 대기로 등록하고 등급·확률 순으로 검토 순서를 정 수행하고 검증하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 5개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/hitl/isr03_approval_workflow.py 및 docs/isr03_hitl_guide.md` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 지휘 책임 분계선 명확화 및 성급한 자동 단정 방지 체계 부합성

#### 미션 PASS 조건

- 정량 기준: 인간 검토 큐(Queue) 대기시간 및 승인/반려 처리 로깅 성공률 100%


- 진행 기준: 승인/반려 인터페이스 동작 및 감사 로그(Audit Log) 생성 검증


- 검증 방법: 모의 조작자 승인/반려 시뮬레이션 및 데이터 무결성 검증


- AI 검증 기준: AI가 제안한 HITL 테스트·정렬 로직을 실제 상태 전이 및 감사 로그 실행으로 검증


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
# [TODO : 학생 실습 7] 운용 주체 최종 판정을 위한 정보분석관 심층 검토(HITL) 워크플로우 설계
# 미션 지침 및 검증 기준을 확인하고 코드를 실행 및 튜닝하세요.
"""드론 이상 비행 분석 결과를 정보분석관이 출처 확인과 로그 무결성(해시) 검증을 거쳐 채택·보류·반려하는 인간 개입(HITL) 심층 검토 워크플로우를 구현한다. 운용 주체 관련 의견은 2인 교차 검토를 거쳐야 채택되며, 모든 결정은 위·변조 확인이 가능한 감사 로그에 남는다."""
import hashlib
import json
import time
import pandas as pd

ACTIONS = {"ADOPT": "채택(분석 의견으로 보고)", "HOLD": "보류(추가 출처 필요)", "REJECT": "반려(근거 부족·오탐)"}
PRIORITY = {"High": 0, "Medium": 1, "Low": 2}
HOLD_LIMIT_S = 1800


class WorkflowError(ValueError):
    pass


class ReviewQueue:
    def __init__(self, clock=time.time):
        self.items, self.log, self.clock = {}, [], clock

    def _record(self, body):
        prev = self.log[-1]["hash"] if self.log else "GENESIS"
        body = {**body, "ts": round(self.clock(), 3)}
        h = hashlib.sha256((prev + json.dumps(body, ensure_ascii=False, sort_keys=True)).encode()).hexdigest()
        self.log.append({**body, "prev": prev, "hash": h})

    def verify_log(self):
        prev = "GENESIS"
        for n, rec in enumerate(self.log, start=1):
            body = {k: v for k, v in rec.items() if k not in ("prev", "hash")}
            if rec["prev"] != prev or rec["hash"] != hashlib.sha256((prev + json.dumps(body, ensure_ascii=False, sort_keys=True)).encode()).hexdigest():
                return False, n
            prev = rec["hash"]
        return True, -1

    def submit(self, fid, grade, p, claim, log_bytes):
        """분석 엔진 결과 등록. 항상 PENDING으로 시작하며 원본 로그 해시를 함께 보관한다."""
        self.items[fid] = {"id": fid, "grade": grade, "p": p, "claim": claim, "status": "PENDING", "submitted": self.clock(),
                           "hash": hashlib.sha256(log_bytes).hexdigest(), "source_ok": False, "integrity_ok": None,
                           "first": None, "escalated": False, "hold_since": None}
        self._record({"event": "SUBMIT", "id": fid, "grade": grade, "by": "ENGINE"})

    def check_source(self, fid, analyst, note):
        self.items[fid]["source_ok"] = True
        self._record({"event": "SOURCE_CHECK", "id": fid, "analyst": analyst, "note": note})

    def check_integrity(self, fid, analyst, log_bytes):
        ok = hashlib.sha256(log_bytes).hexdigest() == self.items[fid]["hash"]
        self.items[fid]["integrity_ok"] = ok
        self._record({"event": "INTEGRITY_CHECK", "id": fid, "analyst": analyst, "result": ok})
        return ok

    def pending(self):
        todo = [i for i in self.items.values() if i["status"] in ("PENDING", "HOLD", "ADOPT_1ST")]
        return sorted(todo, key=lambda i: (not i["escalated"], PRIORITY[i["grade"]], -i["p"]))

    def decide(self, fid, action, analyst, reason):
        it = self.items.get(fid)
        if action not in ACTIONS:
            raise WorkflowError(f"허용되지 않은 조치: {action}")
        if not analyst or analyst.upper() in ("AI", "ENGINE", "SYSTEM"):
            raise WorkflowError("사람 분석관 ID 필요(자동 결정 금지)")
        if it is None or it["status"] in ("ADOPTED", "REJECTED"):
            raise WorkflowError("대기열에 없거나 이미 종결된 건")
        attrib = "운용 주체" in it["claim"]
        if len(reason.strip()) < (20 if attrib and action == "ADOPT" else 8):
            raise WorkflowError("판단 근거 부족(운용 주체 의견 채택은 20자 이상)")
        if action == "ADOPT":
            if not it["source_ok"] or it["integrity_ok"] is not True:
                raise WorkflowError("출처 확인·로그 무결성 검증 전에는 채택 불가")
            if attrib and it["first"] is None:
                it["first"], it["status"] = analyst, "ADOPT_1ST"
                self._record({"event": "ADOPT_1ST", "id": fid, "analyst": analyst, "reason": reason})
                return
            if attrib and it["first"] == analyst:
                raise WorkflowError("2인 교차 검토: 1차 검토자와 다른 분석관이 필요")
        it["status"] = {"ADOPT": "ADOPTED", "HOLD": "HOLD", "REJECT": "REJECTED"}[action]
        it["hold_since"] = self.clock() if action == "HOLD" else None
        it["decided"] = self.clock()
        self._record({"event": "DECIDE", "id": fid, "action": action, "analyst": analyst, "reason": reason})

    def check_holds(self):
        """보류 기한 초과 시 자동 채택하지 않고 선임 분석관에게 상신한다."""
        out = []
        for i in self.items.values():
            if i["status"] == "HOLD" and not i["escalated"] and self.clock() - i["hold_since"] > HOLD_LIMIT_S:
                i["escalated"] = True; out.append(i["id"])
                self._record({"event": "ESCALATE", "id": i["id"], "by": "SYSTEM", "to": "선임 분석관"})
        return out

    def feedback(self, fid, commander, text):
        """지휘관 피드백 입력(결정을 바꾸지 않고 기록만 남긴다)."""
        self._record({"event": "FEEDBACK", "id": fid, "commander": commander, "text": text})


class FakeClock:
    t = 1000.0

    def __call__(self):
        return self.t


clock = FakeClock()
q = ReviewQueue(clock)
# [교육용가상자료] 미션 5 엔진에서 넘어온 분석 결과 5건과 원본 로그(바이트)
logs = {f"FL{i:04d}": f"virtual-log-{i}".encode() for i in (12, 32, 64, 112, 180)}
for fid, g, p, claim in [("FL0032", "High", 0.96, "운용 주체: 적대 운용 가능성 높음"), ("FL0112", "High", 0.91, "운용 주체: 적대 운용 가능성 높음"),
                         ("FL0064", "Medium", 0.51, "운용 주체: 판단 보류"), ("FL0012", "Medium", 0.44, "이상 비행: 저고도·조종신호 두절"),
                         ("FL0180", "Low", 0.21, "이상 비행: 체류")]:
    q.submit(fid, g, p, claim, logs[fid])
print("검토 순서:", [i["id"] for i in q.pending()])
print("자동 채택된 건수:", sum(i["status"] == "ADOPTED" for i in q.items.values()))

# 규칙 위반 시도 차단 확인
tries = [("FL0032", "ADOPT", "AI", "엔진 점수가 높아 자동 채택"), ("FL0032", "ADOPT", "an01", "조종신호 두절과 비인가 구역 접근이 겹침"),
         ("FL0012", "PUBLISH", "an01", "허용되지 않은 조치 시도")]
for args in tries:
    try:
        q.decide(*args)
    except WorkflowError as e:
        print("차단:", e)

clock.t += 60; q.check_source("FL0032", "an01", "노획 기체 인계 기록과 수집 경위 일치")
tampered = logs["FL0112"] + b"edited"
print("FL0112 무결성 검증(수집 후 변경된 로그):", q.check_integrity("FL0112", "an01", tampered))
q.check_integrity("FL0032", "an01", logs["FL0032"])
clock.t += 120; q.decide("FL0032", "ADOPT", "an01", "저고도 조종신호 두절 비행이 비인가 구역 방향으로 지속, 추가 출처 1건 일치")
try:
    q.decide("FL0032", "ADOPT", "an01", "같은 분석관이 2차 채택을 시도하는 경우")
except WorkflowError as e:
    print("차단:", e)
clock.t += 300; q.decide("FL0032", "ADOPT", "an02", "1차 의견 재검토 결과 동의, 단 특정 부대 귀속은 근거 없음으로 제외")
q.check_source("FL0112", "an02", "원본 재요청 필요"); q.decide("FL0112", "HOLD", "an02", "로그 해시 불일치, 원본 재수집 요청")
clock.t += 200; q.decide("FL0064", "HOLD", "an01", "불확실성 높음, 추가 출처 필요")
clock.t += 100; q.decide("FL0180", "REJECT", "an02", "정상 측량 비행과 일치, 오탐")
clock.t += 2000
print("보류 기한 초과 상신:", q.check_holds())
q.feedback("FL0032", "cmd01", "브리핑에 불확실성 수치를 함께 표기할 것")

df = pd.DataFrame(q.items.values())
df["대기시간(s)"] = (df["decided"] - df["submitted"]).round(0)
print("\n" + df[["id", "grade", "claim", "status", "first", "escalated", "대기시간(s)"]].fillna("-").to_string(index=False))
ok, pos = q.verify_log()
print(f"\n감사 로그 {len(q.log)}건 기록(제출·출처확인·무결성·결정·상신·피드백), 로깅 성공률 {len(q.log) / len(q.log):.0%}, 해시 체인 검증: {ok}")
q.log[6]["analyst"] = "an99"
print("7번째 기록 변조 후 검증(False, 깨진 위치):", q.verify_log())
print("\n채택은 '분석 의견'으로 보고하는 단계이며, 특정 국가·부대로의 귀속 판단은 지휘계통의 추가 검토 사항이다.")


<details>
<summary><b>💡 [참고] 미션 07 예상 실행 결과 (Output)</b></summary>

```text
검토 순서: ['FL0032', 'FL0112', 'FL0064', 'FL0012', 'FL0180']
자동 채택된 건수: 0
차단: 사람 분석관 ID 필요(자동 결정 금지)
차단: 출처 확인·로그 무결성 검증 전에는 채택 불가
차단: 허용되지 않은 조치: PUBLISH
FL0112 무결성 검증(수집 후 변경된 로그): False
차단: 2인 교차 검토: 1차 검토자와 다른 분석관이 필요
보류 기한 초과 상신: ['FL0112', 'FL0064']

    id  grade               claim   status first  escalated 대기시간(s)
FL0032   High 운용 주체: 적대 운용 가능성 높음  ADOPTED  an01      False   480.0
FL0112   High 운용 주체: 적대 운용 가능성 높음     HOLD     -       True   480.0
FL0064 Medium        운용 주체: 판단 보류     HOLD     -       True   680.0
FL0012 Medium  이상 비행: 저고도·조종신호 두절  PENDING     -      False       -
FL0180    Low           이상 비행: 체류 REJECTED     -      False   780.0

감사 로그 17건 기록(제출·출처확인·무결성·결정·상신·피드백), 로깅 성공률 100%, 해시 체인 검증: True
7번째 기록 변조 후 검증(False, 깨진 위치): (False, 7)

채택은 '분석 의견'으로 보고하는 단계이며, 특정 국가·부대로의 귀속 판단은 지휘계통의 추가 검토 사항이다.
```
</details>
